# Testing an adjoint for sensors between grid points
Build a forward instrument response from smoothing and interpolation. Derive its transpose by reversing the operations and by scattering a residual to both neighbouring grid nodes. The notebook catches a plausible wrong adjoint using inner products and a Taylor test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cells=12
fraction=.35                     # position between two grid nodes
assert 6 <= cells <= 100 and .05 <= fraction <= .95
rng=np.random.default_rng(4)
I=np.eye(cells)
S=.6*I+.25*np.roll(I,1,axis=1)+.15*np.roll(I,-1,axis=1)
left=np.arange(0,cells,2)
B=np.zeros((len(left),cells))
B[np.arange(len(left)),left]=1-fraction
B[np.arange(len(left)),(left+1)%cells]=fraction
A=B@S
m=rng.normal(size=cells);direction=rng.normal(size=cells)
y=rng.normal(size=len(left));obs=rng.normal(size=len(left))
def forward(v):return B@(S@v)
def adjoint(w):return S.T@(B.T@w)
def wrong(w):return S@(B.T@w)   # the smoothing stencil is not symmetric
gap=abs(forward(direction)@y-direction@adjoint(y))
assert gap < 1e-12
print("Dot-product error with exact transpose:",gap)
print("Dot-product error with unreversed smoothing:", abs(forward(direction)@y-direction@wrong(y)))

## Use an impulse and a Taylor-remainder test
A single sensor residual shows which parameter nodes it influences. The cost is half the squared measurement error. Subtract the predicted directional derivative and examine the remainder as the step is halved.

In [ ]:
fig,(footprint,rate)=plt.subplots(1,2,figsize=(10,4),layout="constrained")
sensor=np.zeros(len(left));sensor[len(left)//2]=1
n=np.arange(cells)
footprint.plot(n,adjoint(sensor),"o-",color="#145c71",label="adjoint")
footprint.plot(n,wrong(sensor),"s--",ms=4,color="#b94626",label="unreversed stencil")
footprint.set(xlabel="Parameter node",ylabel="Sensitivity",title="Unit residual at one sensor")
footprint.legend()
def cost(v):return np.linalg.norm(forward(v)-obs)**2/2
residual=forward(m)-obs
d1=direction@adjoint(residual); d2=direction@wrong(residual)
hh=2.**(-np.arange(2,17))
for d,c,label in [(d1,"#145c71","exact transpose"),(d2,"#b94626","wrong transpose")]:
    rate.loglog(hh,[abs(cost(m+h*direction)-cost(m)-h*d) for h in hh],"o-",ms=3,color=c,label=label)
rate.set(xlabel="Perturbation size h",ylabel="Absolute Taylor remainder",title="Second order or first order?")
rate.legend()
for ax in (footprint,rate):ax.grid(alpha=.2)
plt.show()

## Check the coefficient as well as the slope
For a linear measurement the exact remainder is half the squared length of the change in the predicted measurements. This supplies a stronger independent test than inspecting a graph by eye.

In [ ]:
h=.02
remainder=cost(m+h*direction)-cost(m)-h*d1
assert np.isclose(remainder,.5*h*h*(forward(direction)@forward(direction)),atol=1e-12)
print("Remainder at h = 0.02:",remainder)